# Exploratory Data Analysis Organized

In [2]:
print('Kernel test 1234')

Kernel test 1234


In [3]:
# Step 0, lets import the regular stuff we will probably need
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (ModelSpec as MS,
summarize)
print('Done part 1212323')

# MORE MOUSE BITES
# Ehh, we should really start learning SCIKITLEARN next or whatever its called
# but I think for this project its not horrible to stick with the ISLP resources
# since we ARE straying from the notes now
from ISLP import confusion_table
from ISLP.models import contrast
from sklearn.discriminant_analysis import \
(LinearDiscriminantAnalysis as LDA ,
QuadraticDiscriminantAnalysis as QDA)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt
import re
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

print('Done part 2 with gitt')

Done part 1212323
Done part 2 with gitt


In [61]:
# Import the data, this time we also import the test data.
Titanic_train = pd.read_csv('titanic_data/train.csv')
Titanic_test = pd.read_csv('titanic_data/test.csv')

Titanic_test.head()
#Titanic_train.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


The plan is this:

 1(a)Drop 'PassengerId' column

 1(b) Create column called 'Title' to estimate the missing ages. Estimate missing ages

 1(c) Combine SibSp and ParCh columns into one (SibSp + ParCh). Then make it into bins (we dont want bins that are too small, to avoid overfitting). Drop 'SibSP', 'ParCh' columns as well as any created temporarily to calculate this Family_Bins style column

 1(d) Make 'Has_Cabin_Record' column. Drop 'Cabin' column 

 1(e) Use ticket to calculate log+1(Fare). Create 'log_plus1_Fare' column. Drop 'Fare' and 'Ticket' columns.

 1(f) Delete the rows where 'Embarked' is null. Setup so that for future training, we assume Embarked = Mode (S?). 
 
 1(f2) For train data specifically, we fill missing 'Embarked' values in the test set with 'S'

 1(g) Drop columns 'Name' and 'Title'

 1(h) Change the data type of some of the columns to the 'category' data type to make them easier to manipulate.

In [62]:
#1(a)

# We drop the useless looking column
Titanic_train = Titanic_train.drop(columns='PassengerId')
Titanic_test = Titanic_test.drop(columns='PassengerId')

In [63]:
# 1(b)

# Then we use regex to find the different groups, and make a new column. The column we care about is 'Name', we categorize according to cat we found and what google says they mean
conditions_train = [
    Titanic_train['Name'].str.contains(r'Master') 
    , Titanic_train['Name'].str.contains(r'Mrs.') | Titanic_train['Name'].str.contains(r'Mme.')# Married woman Mme
    , Titanic_train['Name'].str.contains(r'Mr.') # Mr.
    , Titanic_train['Name'].str.contains(r'Miss.') | Titanic_train['Name'].str.contains(r'Mlle.') # Girl or unmarried woman, Unmarried
]

# Test Titles (Must check Titanic_test['Name']!)
conditions_test = [
    Titanic_test['Name'].str.contains(r'Master'),
    Titanic_test['Name'].str.contains(r'Mrs.') | Titanic_test['Name'].str.contains(r'Mme.'),
    Titanic_test['Name'].str.contains(r'Mr.'),
    Titanic_test['Name'].str.contains(r'Miss.') | Titanic_test['Name'].str.contains(r'Mlle.')
]

choices_name = ['Master'
                , 'Mrs/Mme'
                , 'Mr'
                , 'Miss/Mlle']

Titanic_train['Title'] = np.select(conditions_train, choices_name, default='Other')
Titanic_test['Title'] = np.select(conditions_test, choices_name, default='Other')


# Now we estimate ages that are missing

# We review the value counts, Other is a BIT suspect in terms of being too small a category, but since 
# we are using this for age estimation as opposed to category, I think this is ok
#Titanic_train['Title'].value_counts()

# Thanks Gemini for rewriting this part. 

# Step A: Compute median ages strictly from Titanic_train
train_title_medians = Titanic_train.groupby('Title')['Age'].median()

# Step B: Fill missing ages in Titanic_train using its own medians
Titanic_train['Age'] = Titanic_train['Age'].fillna(
    Titanic_train['Title'].map(train_title_medians)
)

# Step C: Fill missing ages in Titanic_test using TRAIN medians (No Leakage!)
Titanic_test['Age'] = Titanic_test['Age'].fillna(
    Titanic_test['Title'].map(train_title_medians)
)

In [64]:
# 1(c)
Titanic_train['Family'] = Titanic_train['SibSp']+Titanic_train['Parch']
Titanic_test['Family'] = Titanic_test['SibSp']+Titanic_test['Parch']

# Create bins for SibSp and ParCh
bins_01_2 = [-0.5, 0.5, 1.5, 1000]
labels_01_2 = ['Zero', 'One', 'Two or more']

# Create bins for "Family"
bins_012_3 = [-0.5, 0.5, 1.5, 2.5, 1000]
labels_012_3 = ['Zero', 'One', 'Two', 'Three or more']

Titanic_train['Sibsp_bins'] = pd.cut(Titanic_train['SibSp'], bins_01_2, labels = labels_01_2)
Titanic_train['Parch_bins'] = pd.cut(Titanic_train['Parch'], bins_01_2, labels = labels_01_2)
Titanic_train['Family_bins'] = pd.cut(Titanic_train['Family'], bins_012_3, labels = labels_012_3)


Titanic_test['Sibsp_bins'] = pd.cut(Titanic_test['SibSp'], bins_01_2, labels = labels_01_2)
Titanic_test['Parch_bins'] = pd.cut(Titanic_test['Parch'], bins_01_2, labels = labels_01_2)
Titanic_test['Family_bins'] = pd.cut(Titanic_test['Family'], bins_012_3, labels = labels_012_3)
Titanic_test.head(10)

# Drop actual values. Bins to be decided later
Titanic_train = Titanic_train.drop(columns='SibSp')
Titanic_train = Titanic_train.drop(columns='Parch')
Titanic_train = Titanic_train.drop(columns='Family')

Titanic_test = Titanic_test.drop(columns='SibSp')
Titanic_test = Titanic_test.drop(columns='Parch')
Titanic_test = Titanic_test.drop(columns='Family')

In [65]:
# 1(d)
Titanic_train['Has_Cabin_Record'] = Titanic_train['Cabin'].notna().astype(int)
Titanic_test['Has_Cabin_Record'] = Titanic_test['Cabin'].notna().astype(int)

Titanic_train = Titanic_train.drop(columns='Cabin')
Titanic_test = Titanic_test.drop(columns='Cabin')

In [66]:
# 1(e)

ticket_counts_train = Titanic_train['Ticket'].value_counts()
ticket_counts_test = Titanic_test['Ticket'].value_counts()

# It turns out theres a missing Fare value, so we fill that in with the median
# Calculate median Fare strictly from the training set
train_fare_median = Titanic_train['Fare'].median()

# Fill missing Fare in test set with the train median
Titanic_test['Fare'] = Titanic_test['Fare'].fillna(train_fare_median)

group_size_by_ticket_train_train = Titanic_train['Ticket'].map(ticket_counts_train)
group_size_by_ticket_train_test = Titanic_test['Ticket'].map(ticket_counts_test)


Titanic_train['Individual_Fare_LogPlus1'] = np.log1p(Titanic_train['Fare'] / group_size_by_ticket_train_train)
Titanic_test['Individual_Fare_LogPlus1'] = np.log1p(Titanic_test['Fare'] / group_size_by_ticket_train_test)


Titanic_train = Titanic_train.drop(columns='Fare')
Titanic_train = Titanic_train.drop(columns='Ticket')

Titanic_test = Titanic_test.drop(columns='Fare')
Titanic_test = Titanic_test.drop(columns='Ticket')

In [67]:
# 1(f)
Titanic_train = Titanic_train.dropna(subset=['Embarked'])

# 1(g) (For test data:) Fill missing 'Embarked' values in the test set with 'S'
Titanic_test['Embarked'] = Titanic_test['Embarked'].fillna('S')

In [68]:
# 1(g) 
Titanic_train = Titanic_train.drop(columns='Name')
Titanic_train = Titanic_train.drop(columns='Title')

Titanic_test = Titanic_test.drop(columns='Name')
Titanic_test = Titanic_test.drop(columns='Title')

In [69]:
# 1(h) 
# Change the data type of some of the columns to the 'category' data type to make them easier to manipulate.
# I'm not sure this part matters for the training set, as maybe we only need this bit to fit the model. 
# However, its dead easy to implement and doesnt hurt, so Im just gonna add it without thinking much more
Titanic_train['Sex'] = Titanic_train['Sex'].astype('category')
Titanic_test['Sex'] = Titanic_test['Sex'].astype('category')

Titanic_train['Embarked'] = Titanic_train['Embarked'].astype('category')
Titanic_test['Embarked'] = Titanic_test['Embarked'].astype('category')

# Force Family_bins to be an UNORDERED category so MS() generates dummy variables. Thank you Gemini
Titanic_train['Family_bins'] = pd.Categorical(
    Titanic_train['Family_bins'], 
    categories=['Zero', 'One', 'Two', 'Three or more'], 
    ordered=False
)

Titanic_test['Family_bins'] = pd.Categorical(
    Titanic_test['Family_bins'],
    categories=['Zero', 'One', 'Two', 'Three or more'],
    ordered=False
)

# Again thank you Gemini for this change in datatype.
Titanic_train['Sibsp_bins'] = pd.Categorical(Titanic_train['Sibsp_bins'], ordered=False)
Titanic_train['Parch_bins'] = pd.Categorical(Titanic_train['Parch_bins'], ordered=False)
Titanic_train['Pclass']=pd.Categorical(Titanic_train['Pclass'], ordered=False)

Titanic_test['Sibsp_bins'] = pd.Categorical(Titanic_test['Sibsp_bins'], ordered=False)
Titanic_test['Parch_bins'] = pd.Categorical(Titanic_test['Parch_bins'], ordered=False)
Titanic_test['Pclass']=pd.Categorical(Titanic_test['Pclass'], ordered=False)

In [70]:
# 1(i)
# Now we drop the SiblingSpouse and ParentChild columns (since we have Family)
Titanic_train = Titanic_train.drop(columns='Sibsp_bins')
Titanic_train = Titanic_train.drop(columns='Parch_bins')

Titanic_test = Titanic_test.drop(columns='Sibsp_bins')
Titanic_test = Titanic_test.drop(columns='Parch_bins')

In [71]:
# 1(j) Create an interaction column between Sex and Pclass. Then Create Age Squared

# Interaction term between Sex and PClass on training data
Titanic_train['Sex_Pclass_Interaction'] = Titanic_train['Sex'].astype(str).str.cat(
    'P' + Titanic_train['Pclass'].astype(str), 
    sep='_'
)

# Now same thing but on test data
Titanic_test['Sex_Pclass_Interaction'] = Titanic_test['Sex'].astype(str).str.cat(
    'P' + Titanic_test['Pclass'].astype(str), 
    sep='_'
)


# Create Age Squared on training data
Titanic_train['Age_Squared'] = Titanic_train['Age'] ** 2

# Create Age Squared on test data
Titanic_test['Age_Squared'] = Titanic_test['Age'] ** 2

In [72]:
# 1(k), 2 new interactions: Sex_Age and PClass

# 1. Sex x Age Interaction
is_male_train = (Titanic_train['Sex'] == 'male').astype(int)
Titanic_train['Sex_male_x_Age'] = is_male_train * Titanic_train['Age']

is_male_test = (Titanic_test['Sex'] == 'male').astype(int)
Titanic_test['Sex_male_x_Age'] = is_male_test * Titanic_test['Age']

# 2. Pclass_3 x Fare Interaction
is_p3_train = (Titanic_train['Pclass'] == 3).astype(int)
Titanic_train['Pclass3_x_Fare'] = is_p3_train * Titanic_train['Individual_Fare_LogPlus1']

is_p3_train = (Titanic_test['Pclass'] == 3).astype(int)
Titanic_test['Pclass3_x_Fare'] = is_p3_train * Titanic_test['Individual_Fare_LogPlus1']

In [73]:
# 1(l) Drop some pruned columns using
cols_to_drop = ['Individual_Fare_LogPlus1', 'Pclass']

Titanic_train = Titanic_train.drop(columns='Individual_Fare_LogPlus1')
Titanic_test = Titanic_test.drop(columns='Individual_Fare_LogPlus1')

Titanic_train = Titanic_train.drop(columns='Pclass')
Titanic_test = Titanic_test.drop(columns='Pclass')

Titanic_test.head()

,Sex,Age,Embarked,Family_bins,Has_Cabin_Record,Sex_Pclass_Interaction,Age_Squared,Sex_male_x_Age,Pclass3_x_Fare
0,male,34.5,Q,Zero,0,male_P3,1190.25,34.5,2.178064
1,female,47.0,S,One,0,female_P3,2209.00,0.0,2.079442
2,male,62.0,Q,Zero,0,male_P2,3844.00,62.0,0.000000
3,male,27.0,S,Zero,0,male_P3,729.00,27.0,2.268252
4,female,22.0,S,Two,0,female_P3,484.00,0.0,2.586824


In [74]:
# Part used to train the model, on the training data only!

def Model_Fitter_Sklearn(dataframe, target_col='Survived', model_type='logistic', cv_folds=5):
    """
    Fits Logistic Regression, LDA, or QDA models, evaluates CV accuracy,
    and returns a summary DataFrame alongside the fitted model.
    """
    
    # 1. Separate features and target
    X_raw = dataframe.drop(columns=[target_col])
    y = dataframe[target_col]
    
    # 2. One-hot encode categorical features (dummy encoding with baseline)
    X = pd.get_dummies(X_raw, drop_first=True)
    
    # 3. Instantiate model
    model_type = model_type.lower()
    if model_type == 'logistic':
        model = LogisticRegression(max_iter=5000)
    elif model_type == 'lda':
        model = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
    elif model_type == 'qda':
        model = QuadraticDiscriminantAnalysis(reg_param=0.1)
    elif model_type in ['nb', 'naive_bayes']:
        model = GaussianNB()
    else:
        raise ValueError("Invalid model_type! Choose 'logistic', 'lda', 'qda', or 'nb'")
    
    # 4. Perform K-Fold Cross Validation
    cv_scores = cross_val_score(model, X, y, cv=cv_folds, scoring='accuracy')
    
    # 5. Fit the final model on all provided training data. This 'fit' part again seems a bit superfluous, how could we have calculated the
    # cv_scores without having fit the models b4? Seems like I'm Double Fittig? idk, whatever
    model.fit(X, y)
    train_acc = model.score(X, y)
    variance_gap = train_acc - cv_scores.mean()
    
    # 6. Print CV metrics, plus training accuracy and the "variance gap", i.e. the difference between train_acc and cv_scores.
    print("=" * 55)
    print(f"Model Type: {model_type.upper()}")
    print(f"Training Set Accuracy:       {train_acc:.4f}")
    print(f"{cv_folds}-Fold CV Accuracy Scores: {cv_scores.round(4)}")
    print(f"Mean CV Accuracy:           {cv_scores.mean():.4f}")
    print(f"Variance Gap (Train - CV):   {variance_gap:.4f}")
    print("=" * 55)
    
    # 7. Construct Summary Table
    # ------------------------------------------------------------------
    # CASE A: Linear Models (Logistic Regression & LDA) -> Return Coefficients
    # ------------------------------------------------------------------
    if hasattr(model, 'coef_'):
        summary_df = pd.DataFrame({
            'Feature': X.columns,
            'Coefficient (Log-Odds)': model.coef_[0].round(4),
            'Odds Ratio': np.exp(model.coef_[0]).round(4)
        })
        
        # Add intercept if available
        if hasattr(model, 'intercept_'):
            intercept_val = model.intercept_[0]
            intercept_row = pd.DataFrame({
                'Feature': ['Intercept'],
                'Coefficient (Log-Odds)': [np.round(intercept_val, 4)],
                'Odds Ratio': [np.round(np.exp(intercept_val), 4)]
            })
            summary_df = pd.concat([intercept_row, summary_df], ignore_index=True)
            
    # ------------------------------------------------------------------
    # CASE B: Naive Bayes - Return Coefficients
    # ------------------------------------------------------------------        
    elif hasattr(model, 'theta_'):  # Naive Bayes class means
        means_df = pd.DataFrame(model.theta_.T, columns=['Mean (Perished: Y=0)', 'Mean (Survived: Y=1)'])
        means_df.insert(0, 'Feature', X.columns)
        means_df['Mean (Perished: Y=0)'] = means_df['Mean (Perished: Y=0)'].round(4)
        means_df['Mean (Survived: Y=1)'] = means_df['Mean (Survived: Y=1)'].round(4)
        means_df['Absolute Difference'] = (means_df['Mean (Survived: Y=1)'] - means_df['Mean (Perished: Y=0)']).abs().round(4)
        
        print(f"\nNB Class Priors (pi_k): Perished={model.class_prior_[0]:.4f}, Survived={model.class_prior_[1]:.4f}")
        summary_df = means_df
    
    # ------------------------------------------------------------------
    # CASE C: Quadratic Model (QDA) -> Return Class Priors & Class Means
    # ------------------------------------------------------------------
    else:
        # Create a summary of class means (mu_k) for each feature
        means_df = pd.DataFrame(model.means_.T, columns=['Mean (Perished: Y=0)', 'Mean (Survived: Y=1)'])
        means_df.insert(0, 'Feature', X.columns)
        
        # Round the values for readability
        means_df['Mean (Perished: Y=0)'] = means_df['Mean (Perished: Y=0)'].round(4)
        means_df['Mean (Survived: Y=1)'] = means_df['Mean (Survived: Y=1)'].round(4)
        
        # Calculate the absolute difference to see which features separate classes best
        means_df['Absolute Difference'] = (means_df['Mean (Survived: Y=1)'] - means_df['Mean (Perished: Y=0)']).abs().round(4)
        
        print(f"\nQDA Class Priors (pi_k): Perished={model.priors_[0]:.4f}, Survived={model.priors_[1]:.4f}")
        summary_df = means_df
        
    return model, summary_df

In [76]:
# Now the best result was from using df_logistic_ttf4 (see Titanic_Notebook_EDA_rough.ipynb), 
# basically Logistic regression with a subselection of columns including some new interactions
final_model , summary_df = Model_Fitter_Sklearn(
    Titanic_train, target_col = "Survived", model_type="logistic", cv_folds = 5)

X_test_raw = Titanic_test.copy()
X_train_raw = Titanic_train.drop(columns="Survived")

X_train = pd.get_dummies(X_train_raw, drop_first=True)
X_test = pd.get_dummies(X_test_raw, drop_first=True)

test_predictions = final_model.predict(X_test)

submission = pd.DataFrame({
    'PassengerID': Titanic_test.index,
    'Survived': test_predictions
})

# 7. Save to CSV without index
submission.to_csv('submission.csv', index=False)
print("Submission file successfully created!")

Model Type: LOGISTIC
Training Set Accuracy:       0.8324
5-Fold CV Accuracy Scores: [0.8371 0.8258 0.8371 0.8146 0.8644]
Mean CV Accuracy:           0.8358
Variance Gap (Train - CV):   -0.0034
Submission file successfully created!


In [77]:
# Sanity check the proportion of survival
sub = pd.read_csv('submission.csv')
print(f"Predicted Survival Rate: {sub['Survived'].mean():.4f} ({sub['Survived'].sum()}/{len(sub)})")

Predicted Survival Rate: 0.3445 (144/418)
